# 00 - Provide Paths to Relevant Files

OpenAI API Key Path

In [1]:
api_key_path = "/Users/cu135/Library/CloudStorage/OneDrive-Personal/OneDrive_Documents/Work/Software/OpenAI/cu135_cbct_key.txt"

json_file_path is the path to the segmented JSON generated at the end of notebook 03. 

In [2]:
json_file_path = "/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/json/_other_labeled_sections.json"

# 01 - Define Inclusion/Exclusion Questions

Examples generated below:

In [3]:
from calvin_utils.gpt_sys_review.examples.question_utils import QuestionTemplate
question_template = QuestionTemplate()
question_template.inclusion_exclusion_questions()

Here are example inclusion questions:
{
    "Amnesia case report? (Y/N)": "case_report",
    "Published in English? (Y/N)": "is_english"
}
Here are example exclusion questions:
{
    "Transient amnesia, reversible amnesia symptom, severe confabulation or drug use, toxicity, epilepsy-related confusion, psychological or psychiatric-related amnesia (functional amnesia)": "other_cause",
    "Did not examine/report both retrograde and anterograde memory domains": "not_both_domains",
    "Without descriptive/qualitative/quantitative data on amnesia severity/memory tests/questions/scenarios/details": "not_enough_information",
    "Had global cognitive impairment disproportionate to memory loss": "disproportionate_impairment",
    "Without measurable lesion-related brain MR/CT scans": "no_scan",
    "Had focal or widespread brain atrophy": "neurodegenerative",
    "Atypical cases with selective (e.g., semantic) memory loss or material/topographic-specific memory loss": "atypical_case"
}
Here i

**Critical Note**
- You are going to define a dictionary with questions as keys (first) and outcomes as values (second).
- The value determines if you are answering a positive question or a negative question.
- If the question is positive (a yes is good), set the value to 1.
- If the question is negative (a yes is bad), set the value to 0.
- A good paper will be denoted by 1, with a bad paper denoted by 0.

**Examples**
```
question = {
    "Prioritizing implicit and explicit information, do you think this is a ______ case report or case series? (Yes/No)": 1,

    "Prioritizing implicit and explicit information, do you think this is a case of transient amnesia, reversible amnesia, confabulation, epilepsy, toxicity, neurodegenerative disease, or functional/psychiatric neurological disorder? . (Yes/No)": 0,

    "Prioritizing implicit and explicit information, do you think this examined both retrograde and anterograde amnesia? For example, if they report scores or a clinical examination examining retrograde and anterograde amnesia. (Yes/No)": 1,

    "Prioritizing implicit and explicit information, do you think this has some sort of qualitative or quantitative measurments on memory severity? For example, a case with neuropsychological measurements on memory tests. (Yes/No)": 1,

    "Prioritizing implicit and explicit information, do you think the memory loss might be due to global cognitive impairment? For example, a stroke resulting in executive, memory, language, and more changes is a global impairment. (Yes/No)": 0,

    "Prioritizing implicit and explicit information, do you think this case report has a figure with neuroimaging? (Yes/No)": 1,

    "Prioritizing implicit and explicit information, do you think this case report had brain atrophy, either focally or globally suggesting neurodegeneration? (Yes/No)": 0,

    "Prioritizing implicit and explicit information, do you think this is due to atypical memory loss, where only a subset of memory is impaired? For example, if just spatial memory is lost. (Yes/No)": 0,
    
    "Prioritizing implicit and explicit information, do you think this in English? (Yes/No)": 0,
}
```

In [4]:
question = {
"Is the primary outcome of this manuscript about memory? (Yes/No)": 1,
"Does this manuscript use transcranial magnetic stimulation (TMS)? (Yes/No)": 1,
"Does this manuscript report memory outcomes? (Yes/No)": 1}

# 02 - Ask Questions

Define the segmented labels you want to consider. 

- Article type 'case' will has sections 'case_report' and 'other'
- Article type 'research' has sections "Abstract", "Introduction", "Methods", "Results", "Discussion", "Conclusion", "References"
- Article type 'inclusion' can use any keyword you want. It constrains the system to respond in a binary fashion: pass or fail for each question

In [5]:
# Define the keys you want to consider (exclude 'References')
keys_to_consider = [ "Positive", "other" ]  # Add or remove keys as per your requirement
article_type = 'inclusion'

Set test_mode=True during your first few runs, while you tune your questions to get the answers you need
- Always run this first, at least once. 

In [6]:
test_mode=False

Submit Questions

In [7]:
from calvin_utils.gpt_sys_review.gpt_utils.openai_json_evaluator import OpenAIJsonEvaluator
evaluator = OpenAIJsonEvaluator(api_key_path=api_key_path, json_file_path=json_file_path, keys_to_consider=keys_to_consider, question_type=article_type, model_choice="gpt3_small",  question=question, test_mode=test_mode)
answers = evaluator.evaluate_all_files()
new_json_path = evaluator.save_to_json(answers)

100%|██████████| 82/82 [14:43<00:00, 10.77s/it]

Saved to: /Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/json/../case_extractions/inclusion_evaluations.json


# 03 - Summarize the Results

In [8]:
from calvin_utils.gpt_sys_review.json_utils import InclusionExclusionSummarizer
summarizer = InclusionExclusionSummarizer(new_json_path, questions=question)
result_df, raw_path, automated_path = summarizer.run()

Your CSV files of filtered manuscripts have been saved to this directory: 
 /Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/json/../case_extractions/inclusion_exclusion_results


/Users/cu135/Library/CloudStorage/OneDrive-Personal/OneDrive_Documents/Work/Software/ReviewPyper/ReviewPyPerVenv/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


# 04 - Optional) Update Master List
- If you have been using a master_list, you can update it with the results from the generated CSVs.

In [ ]:
master_list_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/TMS_studies_influencing_memory/metadata/Updated_TMS_studies_influencing_memory.csv'

In [2]:
from calvin_utils.gpt_sys_review.txt_utils import PostProcessing
PostProcessing.add_raw_results_to_master_list(master_list_path=master_list_path, raw_results_path=raw_path)

Updating master list: 82it [00:00, 884.76it/s]


,study,PMID,DOI,ses,N,Reported Outcome,Pre-Post Memory Effect Size (Cohen's D),Location,List of Coordinates,Side,...,N.1,_.1,experimental_post_mean,experimental_post_stdev,experimental_post_n,PDF_Downloaded,PDF_Path,Is the primary outcome of this manuscript about memory? (Yes/No),Does this manuscript use transcranial magnetic stimulation (TMS)? (Yes/No),Does this manuscript report memory outcomes? (Yes/No)
0,Jackson 2021,34002006,10.1038/s42003-021-02109-x,dlPFC_R,20,Impaired,-0.622389284,dlPFC,"[44, 31, 28]",R,...,20.0,NaN,57.0,5.0,20.0,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,0.0
1,Turriziani 2012,22514525,10.3389/fnhum.2012.00062,NaN,100,Mixed,-0.738548946,dlPFC,"[-44, 31, 28]",L,...,100.0,NaN,57.0,10.0,100.0,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
2,Turriziani 2012,22514525,10.3389/fnhum.2012.00062,NaN,100,Mixed,0.443129368,dlPFC,"[44, 31, 28]",R,...,100.0,NaN,73.0,10.0,100.0,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
3,Turriziani 2012,22514525,10.3389/fnhum.2012.00062,NaN,100,Mixed,0.073854895,dlPFC,"[-44, 31, 28]",L,...,100.0,NaN,80.0,10.0,100.0,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
4,Turriziani 2012,22514525,10.3389/fnhum.2012.00062,NaN,100,Mixed,-0.590839157,dlPFC,"[44, 31, 28]",R,...,100.0,NaN,72.0,10.0,100.0,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,Wu 2020,31884184,10.1016/j.brs.2019.12.020,NaN,12,Improved,NaN,leftDLPFC,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
120,Xiu 2020,32185388,10.1093/schbul/sbaa035,NaN,110,Improved,NaN,dlPFC,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
121,Yang 2019,30795490,10.1016/j.jad.2018.12.102,NaN,52,Improved,NaN,DLPFC,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,1.0,1.0,1.0
122,Zhuo 2019,31190822,10.2147/NDT.S196086,NaN,60,UNSURE,NaN,mPFC,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,True,/Users/cu135/Partners HealthCare Dropbox/Calvi...,0.0,1.0,1.0


Your articles have been completely evaluated and filtered. 

Please check the CSVs in the directory noted above and use the path to the one you would like to use. It will be for your next notebook.
- Enjoy. If this has been helpful, please consider adding Calvin Howard as a collaborator. 
- e: choward12@bwh.harvard.edu